# 3B.2. Voorbeeld examenvragen — lees, voorspel, verbeter

Hieronder staat telkens een (bijna) volledige oplossing voor een examenvraag. Er zitten één of meerdere bugs in die de assert doen falen (of een fout antwoord opleveren) — vaak gaat het om een verkeerd begrip van hoe de code zich gedraagt (bv. verwarring tussen een as in NumPy/Pandas, een voorwaarde die over de verkeerde scope gaat, een berekening die buiten een lus staat) in plaats van één fout symbool. Lees de code, **voorspel** de output, run, en zoek alle fouten voor je de oplossing bekijkt.

## Vraag 1: pandas_oefening

Gegeven is een Pandas `DataFrame`-object `dataframe_in`. Deze functie moet alle rijen verwijderen waarvan de rijsom strikt kleiner is dan de opgegeven `min_gemiddelde`, en het resultaat teruggeven als een nieuwe dataframe in `dataframe_out`.

**Antwoord:** De code gebruikt `dataframe_in.sum(axis=0)`, wat de som per **kolom** berekent (een Series geïndexeerd op de kolomnamen `c1`, `c2`, `c3`), in plaats van `axis=1` voor de som per **rij** — nodig om vervolgens rijen te kunnen selecteren. Omdat de resulterende boolean Series een heel andere index heeft dan de rijen van `dataframe_in`, kan Pandas de twee niet uitlijnen en gooit het een `IndexingError` in plaats van gewoon een fout antwoord te geven. Dit is een klassieke as-verwarring: denk bij `axis=0` vs. `axis=1` steeds na over welke richting je *samenvat* (rij-voor-rij, of kolom-voor-kolom) versus wat je nadien wil *selecteren* (hier: rijen, dus `axis=1`). Fix: gebruik `axis=1`.

In [1]:
import pandas as pd
import numpy as np

def pandas_oefening(dataframe_in, min_gemiddelde):
    dataframe_out = dataframe_in[dataframe_in.sum(axis=1) >= min_gemiddelde]
    return dataframe_out

data = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
df = pd.DataFrame(data, columns=['c1', 'c2', 'c3'])

df_out = pandas_oefening(df, 8)
print(df_out)
assert np.allclose(df_out.values, [[4, 5, 6],
                                    [7, 8, 9]])

   c1  c2  c3
1   4   5   6
2   7   8   9


## Vraag 2

Gegeven: de dataframe `df_in` uit het bestand `heart_disease_data.csv`. Het bestand bevat informatie over hartpatiënten (leeftijd, gender, cholesterolgehalte, maximale hartslag, enz.).

1. Aggregeer de numerieke kolommen op basis van `'ST_Slope'` en `'HeartDisease'`, via de **mediaan**.
2. Filter de dataframe: `'Age'` moet strikt boven de mediaan van die kolom liggen, **en** `'MaxHR'` moet strikt boven 170 zijn, **en** `'ChestPainType'` moet van het type `'NAP'` zijn.
3. Tel het aantal voorkomens van `'Up'` voor `'ST_Slope'` in die gefilterde data.

Bron data: https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/

**Antwoord:** Twee onafhankelijke fouten. (1) De aggregatie gebruikt `.mean()` in plaats van `.median()` — er wordt uitdrukkelijk om de mediaan per groep gevraagd, niet het gemiddelde; voor scheve verdelingen (zoals cholesterol of bloeddruk) geeft dat systematisch andere waarden, zonder dat er een fout optreedt. (2) De drie filtervoorwaarden worden gecombineerd met `|` (OR) in plaats van `&` (AND) — daardoor blijft een rij al behouden zodra ze aan **één** van de drie voorwaarden voldoet, in plaats van aan **alle drie**. Het gevolg is een veel te ruime selectie: in plaats van 6 patiënten met `ST_Slope == 'Up'` in de gefilterde data, blijven er opeens 250 over. Beide fouten leveren op het eerste gezicht een 'normaal' resultaat op (een tabel met getallen, een positief aantal) — je moet de opgave woord voor woord herlezen om te beseffen dat de aggregatiefunctie en het combineren van voorwaarden niet kloppen. Fix: gebruik `.median()` in de groupby, en combineer de drie voorwaarden met `&`.

In [2]:
import pandas as pd
import numpy as np

def vind_antwoorden_in_dataset(df_in):
    aggregatie = df_in.groupby(['ST_Slope', 'HeartDisease']).median(numeric_only=True)

    mediaan = df_in['Age'].median()
    filtered_df = df_in[(df_in['Age'] > mediaan) & (df_in['MaxHR'] > 170) & (df_in['ChestPainType'] == 'NAP')]
    filtered_df2 = filtered_df.select_dtypes(['int64', 'float64'])

    d1 = filtered_df['ST_Slope'].value_counts()['Up']

    return aggregatie, filtered_df2, d1

df = pd.read_csv('heart_disease_data.csv')
a1, a2, a3 = vind_antwoorden_in_dataset(df)
print('Aggregatie:\n', a1)
print('Aantal:', a3)

assert np.allclose(a1.values, [[ 55.  , 130.  , 236.5 ,   0.  , 147.5 ,   1.45],
       [ 60.  , 125.  , 200.  ,   0.  , 127.  ,   2.  ],
       [ 54.  , 130.  , 229.  ,   0.  , 146.  ,   1.  ],
       [ 56.  , 135.  , 220.  ,   0.  , 124.  ,   1.2 ],
       [ 50.  , 130.  , 226.  ,   0.  , 150.  ,   0.  ],
       [ 58.  , 127.  , 192.  ,   0.  , 144.  ,   0.75]])
assert a3 == 6

Aggregatie:
                         Age  RestingBP  Cholesterol  FastingBS  MaxHR  Oldpeak
ST_Slope HeartDisease                                                         
Down     0             55.0      130.0        236.5        0.0  147.5     1.45
         1             60.0      125.0        200.0        0.0  127.0     2.00
Flat     0             54.0      130.0        229.0        0.0  146.0     1.00
         1             56.0      135.0        220.0        0.0  124.0     1.20
Up       0             50.0      130.0        226.0        0.0  150.0     0.00
         1             58.0      127.0        192.0        0.0  144.0     0.75
Aantal: 6


## Vraag 3

Los de volgende vragen op over de Pandas dataframe over panda's.

- `vraag_1`: bereken de gemiddelde lengte van alle panda's met strikt meer dan 2 vlekken.
- `vraag_2`: maak een nieuwe kolom `'bmi'` aan, berekend als $\frac{Gewicht}{Lengte^2}$.
- `vraag_3`: groepeer de panda's per `'Aantal_Vlekken'` en `'Heeft_Staart'` (in die volgorde) en geef het gemiddelde terug voor de numerieke kolommen `'Gewicht'` en `'Lengte'`.
- `vraag_4`: geef een Python-lijst met daarin alle geluiden van de zwangere panda's.

Elke deelvraag werkt op een kopie `dfv` van de originele dataframe.

**Antwoord:** Twee onafhankelijke fouten, in twee verschillende deelfuncties. (1) In `vraag_2` staat de berekening als `(x['Gewicht'] / x['Lengte']) ** 2` — door de haakjes wordt eerst gedeeld en dán in het kwadraat gebracht, terwijl de BMI-formule $\frac{Gewicht}{Lengte^2}$ vraagt om eerst de lengte in het kwadraat te brengen en dáárna te delen. Dat is een klassieke fout door verkeerd gebruik van haakjes/operatorvolgorde — de uitkomst blijft een zinnig ogend getal, maar is een grootteorde te groot. (2) In `vraag_3` staat de groepering als `groupby(['Heeft_Staart', 'Aantal_Vlekken'])`, terwijl de opgave de volgorde `'Aantal_Vlekken', 'Heeft_Staart'` vraagt. Bij een groupby op meerdere kolommen bepaalt de volgorde van de kolommen de volgorde van de (multi-)index in het resultaat — de waarden per combinatie blijven correct, maar de rijen staan in een andere volgorde, waardoor de assert op `.values` faalt. Fix: gebruik `x['Gewicht'] / x['Lengte']**2` (zonder haakjes rond de deling) in `vraag_2`, en `groupby(['Aantal_Vlekken', 'Heeft_Staart'])` in `vraag_3`.

In [3]:
import numpy as np
import pandas as pd

def vraag_1(df_in):
    dfv = df_in.copy()
    gemiddelde_lengte = dfv[dfv['Aantal_Vlekken'] > 2]['Lengte'].mean()
    return gemiddelde_lengte

def vraag_2(df_in):
    dfv = df_in.copy()
    dfv['bmi'] = dfv.apply(lambda x: x['Gewicht'] / x['Lengte']**2, axis=1)
    return dfv

def vraag_3(df_in):
    dfv = df_in.copy()
    gb = dfv.groupby(['Aantal_Vlekken', 'Heeft_Staart']).mean(numeric_only=True)[['Gewicht', 'Lengte']]
    return gb

def vraag_4(df_in):
    dfv = df_in.copy()
    df_f = dfv[dfv.Is_Zwanger == True]['Geluid'].unique()
    return list(df_f)

np.random.seed(42)
aantal = 15
data = {
    'Gewicht': np.random.normal(100, 25, aantal),
    'Leeftijd': np.random.randint(1, 15, aantal),
    'Lengte': np.random.uniform(1.5, 2.0, aantal),
    'Is_Zwanger': np.random.choice([True, False], size=aantal),
    'Aantal_Vlekken': np.random.randint(0, 5, aantal),
    'Geluid': np.random.choice(['Grom', 'Brom', 'Grom', 'Grom', 'Knor'], size=aantal),
    'Heeft_Staart': np.random.choice([True, False], size=aantal)
}
df = pd.DataFrame(data)

print('Vraag 1:', vraag_1(df))
print('Vraag 2 (bmi):', vraag_2(df)['bmi'].values)
print('Vraag 3:\n', vraag_3(df))
print('Vraag 4:', vraag_4(df))

assert np.allclose(vraag_1(df).round(3), 1.757)
assert np.allclose(vraag_2(df)['bmi'].values, [33.1991572 , 32.43535621, 51.1873178 , 35.5385494 , 29.65928823,
       32.85793815, 61.33663868, 45.67088035, 33.61046181, 33.48382731,
       27.13752086, 24.0534817 , 42.12373204, 18.1465063 , 22.46633826])
assert np.allclose(vraag_3(df).values, [[ 70.21556712,   1.65802152],
       [116.19221345,   1.50663248],
       [119.18586823,   1.61544691],
       [ 94.14657608,   1.69270825],
       [ 92.47897508,   1.76512398],
       [104.77953703,   1.86612766],
       [111.477707  ,   1.69006735],
       [100.09761121,   1.68416322]])
assert vraag_4(df) == ['Grom']

Vraag 1: 1.7567159422974001
Vraag 2 (bmi): [33.1991572  32.43535621 51.1873178  35.5385494  29.65928823 32.85793815
 61.33663868 45.67088035 33.61046181 33.48382731 27.13752086 24.0534817
 42.12373204 18.1465063  22.46633826]
Vraag 3:
                                 Gewicht    Lengte
Aantal_Vlekken Heeft_Staart                      
0              False          70.215567  1.658022
               True          116.192213  1.506632
1              False         119.185868  1.615447
               True           94.146576  1.692708
3              False          92.478975  1.765124
               True          104.779537  1.866128
4              False         111.477707  1.690067
               True          100.097611  1.684163
Vraag 4: ['Grom']


## Vraag 4

Gegeven zijn 2 dataframes:
- `machines_df`: de naam van een machine, de looptijd en de omsteltijd.
- `taken_df`: een taaknaam en een tupel met de namen van de machines die nodig zijn om de taak uit te voeren.

De functie `bereken_totale_doorlooptijd()` wordt als lambda-functie uitgevoerd over de kolom `'Combinatie'` in `taken_df` om een nieuwe kolom `'Totale Doorlooptijd'` te maken. Voor elke machine in de combinatie moet de omsteltijd + looptijd van díe machine opgeteld worden bij het totaal.

**Antwoord:** De omsteltijd wordt maar één keer berekend, vóór de lus, en dan gebaseerd op de éérste machine van de combinatie (`taak_combinatie[0]`) — in plaats van voor élke machine apart, binnen de lus. De opgave is nochtans expliciet: 'omsteltijd + looptijd van elke machine opgeteld', dus élke machine in de combinatie draagt zijn eigen omsteltijd bij, niet enkel de eerste. Doordat de functie toch een plausibel getal teruggeeft (het is alleen te laag), valt dit niet op zonder de opgave woord voor woord na te lezen. Fix: bereken `omsteltijd` binnen de lus, per `machine`, en tel `looptijd + omsteltijd` samen op bij elke iteratie.

In [4]:
import pandas as pd
import numpy as np

def bereken_totale_doorlooptijd(taak_combinatie, machines_df):
    totale_doorlooptijd = 0

    for machine in taak_combinatie:
        looptijd = machines_df.loc[machines_df['Machine'] == machine, 'Looptijd'].values[0]
        omsteltijd = machines_df.loc[machines_df['Machine'] == machine, 'Omsteltijd'].values[0]
        totale_doorlooptijd += looptijd + omsteltijd

    return totale_doorlooptijd

machines_data = {'Machine': ['A', 'B', 'C'],
                 'Looptijd': [10, 15, 20],
                 'Omsteltijd': [5, 8, 10]}
machines_df = pd.DataFrame(machines_data)

taken_data = {'Taak': ['Taak1', 'Taak2', 'Taak3'],
              'Combinatie': [('A', 'B'), ('B', 'C'), ('A', 'C')]}
taken_df = pd.DataFrame(taken_data)

taken_df['Totale Doorlooptijd'] = taken_df['Combinatie'].apply(lambda x: bereken_totale_doorlooptijd(x, machines_df))
print(taken_df)
assert np.allclose(taken_df['Totale Doorlooptijd'].values, [38, 53, 45])

    Taak Combinatie  Totale Doorlooptijd
0  Taak1     (A, B)                   38
1  Taak2     (B, C)                   53
2  Taak3     (A, C)                   45
